# step 1 — pool-500 (RQ2 층 스윕 + K/V 분해 + v 코사인)

**대응 RQ:** RQ2 — 매개 신호가 형태인가, **어느 층**인가, **어느 경로(Key/Value)**인가. (단일 층 인과는 stepC.)

**두 갈래:**
- **A. 개입 측 (인과).** stepC의 L25 KV 치환을 **전 층에 반복** + 각 층 **Key만/Value만/Key+Value** 분해. (층×kind)마다 회복률. donor 2종(`compliant` 주효과 / `unrelated_camel` 형태 통제).
- **B. 관측 측 (방향).** 같은 이름 **camel판 v vs snake판 v의 층별 코사인**. 크기(‖v‖)는 안 변해도 방향이 갈리는지. Value 회복 피크 층과 겹치면 축 B(Value 경로) 확정.

**파일럿 대비 (POOL 확장):** 이름 창고 80→**504(50×50)**. 파일럿은 seed로 12개 랜덤 표집이었으나, scaleup은 **블록**으로 500+개 위반 이름을 커버. seed는 위치·donor 변주.

**코드 동일 보장(재현성):** 개입·측정은 파일럿과 **똑같은 `run(...)`·`run(mode='vcosine')`**. 단 POOL은 표집이라 파일럿(80풀)과 글자 그대로 재현되진 않음 — 새 500 샘플(파일럿은 `results/step1/` 불변).

설계 문서: `docs/step1/scaleup-500.md` (파일럿: `docs/step1/plan.md`). 공유 POOL 코드: `stepB/pool-500`.

> **⚠️ 무겁다:** 스윕은 조건 하나가 **전 층(36) × 3경로** 개입이다. 기본 스윕 **2 donor × 42 block = 84조건** + 코사인 42 = **126조건**(§3 step1 일치). **T4에서 시간이 꽤 걸린다 — 재개 가능.** 무료 티어면 `BLOCKS = list(range(0, 42, 2))`로 21블록(반감).
> **메모리:** output_attentions 안 씀(KV 캐시 편집) → **eager 불필요**, 프롬프트당 1 forward 캐시 재사용.
> **sanity:** 요약에서 S_clean > S_base 확인.

In [ ]:
# 환경 설정 — 설치, GPU 확인, 시드 고정
!pip install -q transformers accelerate torch matplotlib pandas numpy

import random, numpy as np, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (주의: 매우 느림)')

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('seed fixed:', SEED)

In [ ]:
# 저장소 클론 및 브랜치 체크아웃
import os
if not os.path.isdir('HCLT_2026'):
    !git clone https://github.com/deanjs/HCLT_2026.git
%cd HCLT_2026
!git fetch --quiet origin step1/pool-500
!git checkout step1/pool-500
!git pull --quiet origin step1/pool-500
!pip install -e . -q
import sys; sys.path.insert(0, 'src')

In [ ]:
# 조건 설정 — A: 스윕(전 층 x K/V) 2 donor x 블록 / B: 코사인 궤적 블록. 선행 전부 위반(POOL n=0).
from harness.conditions import (Condition, ModelSpec, PrecedingCode, Instruction,
                                Composition, InstructionForm, Notation,
                                Intervention, InterventionKind)
from harness.tasks import NAME_PAIR_POOL

MODEL = ModelSpec(name='Qwen/Qwen2.5-Coder-3B-Instruct', family='qwen', dtype='float16')
DONORS = ['compliant', 'unrelated_camel']   # 음성통제(unrelated_snake)는 stepC에서 확인 -> 생략
N_FUNCTIONS = 12
N_BLOCKS = len(NAME_PAIR_POOL) // N_FUNCTIONS      # 504//12 = 42
BLOCKS = list(range(N_BLOCKS))                     # 무거우면 list(range(0, 42, 2)) 로 21블록
SEEDS = [0]                                        # step1은 무거워 §3이 1~2 권장. 블록이 다양성 제공.

def _pre(block):
    return PrecedingCode(n_compliant=0, n_functions=N_FUNCTIONS,
                         composition=Composition.POOL, pool_block=block)
def _ins():
    return Instruction(form=InstructionForm.POSITIVE, target_notation=Notation.CAMEL)

def sweep_cond(donor, block, s):
    # layers='sweep' -> 전 층. 스윕은 내부에서 key/value/key_value 모두 측정.
    return Condition(model=MODEL, preceding=_pre(block), instruction=_ins(),
                     intervention=Intervention(kind=InterventionKind.KEY_VALUE,
                                               layers='sweep', donor=donor), seed=s)

def cosine_cond(block, s):
    return Condition(model=MODEL, preceding=_pre(block), instruction=_ins(), seed=s, tag='vcosine')

sweep_conditions  = [sweep_cond(d, b, s) for d in DONORS for b in BLOCKS for s in SEEDS]
cosine_conditions = [cosine_cond(b, s) for b in BLOCKS for s in SEEDS]

PREDICTION = ('회복률 피크는 후반부(~L25, 상대 0.6~0.75)에 국소화. compliant와 unrelated_camel '
              '곡선은 전 층에서 겹침(형태 결론 유지). Value 단독 회복 피크 = v 코사인 급락 층 -> 축 B. '
              '500개 위반 이름(블록)에서 이름 무관.')
print(f'풀 {len(NAME_PAIR_POOL)}, {N_BLOCKS}블록 (사용 {len(BLOCKS)})')
print(f'스윕 {len(sweep_conditions)} = {len(DONORS)} donor x {len(BLOCKS)} block x {len(SEEDS)} seed  |  코사인 {len(cosine_conditions)}')

In [ ]:
# 실행 — A: 개입 스윕 / B: 코사인. 조건별 즉시 저장(재개). 로직은 harness가 수행(파일럿과 동일).
from harness import run, ResultRecord, save_result, result_path
from harness.results import load_result
from harness.model import load_model

STEP = 'step1_scaleup500'
handle = load_model(MODEL)   # output_attentions 안 씀 -> eager 불필요
print('layers:', handle.num_layers,
      '| L25 상대 위치:', round(handle.relative_layer(25), 3),
      '| GQA:', handle.gqa_info())

# A. 개입 스윕 (전 층 x K/V)
new = skipped = 0
for i, c in enumerate(sweep_conditions, 1):
    if result_path(c, step=STEP).exists():
        skipped += 1
    else:
        out = run(c, handle=handle)
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2', prediction=PREDICTION))
        new += 1
    if i % 4 == 0 or i == len(sweep_conditions):
        print(f'[스윕 {i}/{len(sweep_conditions)}] 새 {new} / 건너뜀 {skipped}')

# B. 코사인 궤적 (관측)
cnew = cskip = 0
for i, c in enumerate(cosine_conditions, 1):
    if result_path(c, step=STEP).exists():
        cskip += 1
    else:
        out = run(c, handle=handle, mode='vcosine')
        save_result(ResultRecord(condition=out.condition, metrics=out.metrics,
                                 step=STEP, rq='RQ2', prediction=PREDICTION))
        cnew += 1
    if i % 8 == 0 or i == len(cosine_conditions):
        print(f'[코사인 {i}/{len(cosine_conditions)}] 새 {cnew} / 건너뜀 {cskip}')
print('완료.')

In [ ]:
# 결과 로드 — results/step1_scaleup500/
from harness import result_path
from harness.results import load_result

sweep_recs  = [load_result(result_path(c, step=STEP)) for c in sweep_conditions]
cosine_recs = [load_result(result_path(c, step=STEP)) for c in cosine_conditions]
print('로드: 스윕', len(sweep_recs), '/ 코사인', len(cosine_recs), '-> results/'+STEP+'/')

In [ ]:
# 요약 — 층별 회복률 곡선(kind x donor) + 피크 층 + v 코사인 궤적. sanity: S_clean > S_base.
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import defaultdict
from harness.intervention import peak_layer

KINDS = ['key', 'value', 'key_value']

# 층별 회복률: donor -> kind -> {layer: [블록별 회복률]}
agg = {d: {k: defaultdict(list) for k in KINDS} for d in DONORS}
for r in sweep_recs:
    d = r.condition.intervention.donor
    for L, flat in r.metrics.per_layer.items():
        for k in KINDS:
            key = f'{k}__recovery'
            if key in flat:
                agg[d][k][int(L)].append(flat[key])

def curve(d, k):
    layers = sorted(agg[d][k])
    return layers, [float(np.mean(agg[d][k][L])) for L in layers]

print('=== 피크 층 (회복률 최대, 이름 500 집계) ===')
rows = []
for d in DONORS:
    for k in KINDS:
        layers, vals = curve(d, k)
        pk = peak_layer(dict(zip(layers, vals)))
        rows.append({'donor': d, 'kind': k, 'peak_layer': pk[0], 'peak_recovery': round(pk[1], 3)})
print(pd.DataFrame(rows).to_string(index=False))

# v 코사인 궤적 평균
cos = defaultdict(list)
for r in cosine_recs:
    for L, flat in r.metrics.per_layer.items():
        cos[int(L)].append(flat['v_cosine'])
clayers = sorted(cos); cvals = [float(np.mean(cos[L])) for L in clayers]

# 플롯: donor별 회복률 곡선 2 + 코사인 궤적 1
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
colors = {'key': '#2563C9', 'value': '#C6771A', 'key_value': '#2E7D52'}
for j, d in enumerate(DONORS):
    for k in KINDS:
        layers, vals = curve(d, k)
        ax[j].plot(layers, vals, color=colors[k], label=k, marker='.', ms=4)
    ax[j].axhline(0, color='#aaa', lw=.6); ax[j].axhline(1, color='#aaa', ls='--', lw=.6)
    ax[j].axvline(25, color='#B0392B', ls=':', lw=.8)
    ax[j].set_title(f'Recovery sweep — donor={d} (scaleup-500)')
    ax[j].set_xlabel('layer'); ax[j].set_ylabel('recovery'); ax[j].legend(fontsize=8)
ax[2].plot(clayers, cvals, color='#7B3FA0', marker='.', ms=4)
ax[2].axvline(25, color='#B0392B', ls=':', lw=.8)
ax[2].set_title('v cosine trajectory (camel vs snake)')
ax[2].set_xlabel('layer'); ax[2].set_ylabel('cosine (direction)'); ax[2].set_ylim(-0.1, 1.05)
plt.tight_layout(); plt.savefig('step1_scaleup500_summary.png', dpi=110); plt.show()

sc = float(np.mean([r.metrics.extra['S_clean'] for r in sweep_recs]))
sb = float(np.mean([r.metrics.extra['S_base'] for r in sweep_recs]))
print(f'sanity  S_clean {sc:+.2f} > S_base {sb:+.2f} :', sc > sb)

In [ ]:
# 결과 다운로드 — results/step1_scaleup500 을 zip으로 묶어 내려받는다
import shutil
shutil.make_archive('step1_scaleup500_results', 'zip', 'results/'+STEP)
try:
    from google.colab import files
    files.download('step1_scaleup500_results.zip')
except Exception as e:
    print('Colab 아님(수동 다운로드): step1_scaleup500_results.zip', e)